# Andamento dei bond europei e americani

Dati da `data/rendimenti.csv` (aggiornalo con `python src/main.py`). Rendimenti in %, spread in punti percentuali.

- **EUR** = curva dei titoli di Stato AAA dell'area euro (BCE), giornaliera
- **DE/IT/FR/ES** = titoli di Stato per paese a 1 (non per ES), 2, 5, 10 e 30 anni (CNBC): giornalieri dal 2024, settimanali dal 2011, trimestrali dal 2000
- **BCE DE/IT/FR/ES 10Y** = decennali per paese, media mensile (BCE): la storia lunga più precisa, crisi 2011-2012 compresa
- **US** = Treasury (FRED), giornalieri
- **spread corporate** = ICE BofA option-adjusted spread su FRED, solo ultimi 3 anni; **US Baa spread** (Moody's Baa - Treasury 10 anni) dal 2000
- la curva **EUR** (AAA, BCE) parte da settembre 2004

In [1]:
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px

sys.path.insert(0, str(Path.cwd().parent / "src"))
from main import carica

DAL = "2000-01-01"  # inizio dei grafici
df = carica()
linee = lambda cols, titolo, dal=DAL: px.line(df.loc[dal:, cols].dropna(how="all").ffill(), title=titolo,
                                              labels={"value": "%", "Data": "", "variable": ""}).show()
df.index[-1].date()

datetime.date(2026, 11, 2)

## Rendimenti a 10 anni: USA, area euro e paesi

In [2]:
linee(["US 10Y", "BCE DE 10Y", "BCE IT 10Y", "BCE FR 10Y", "BCE ES 10Y"], "Rendimento a 10 anni dal 2000 (paesi euro: media mensile BCE)")
linee(["US 10Y", "EUR 10Y", "DE 10Y", "IT 10Y", "FR 10Y", "ES 10Y"], "Rendimento a 10 anni, ultimi 3 anni", dal="2023-11-01")

## Breve e lungo termine: 2 anni contro 30 anni

In [3]:
for paese in ("US", "EUR", "IT", "DE", "FR", "ES"):
    linee([f"{paese} 2Y", f"{paese} 30Y"], f"{paese}: 2 anni e 30 anni")

## Curva dei rendimenti: oggi, 1, 3 e 10 anni fa
Asse in anni alla scadenza (scala logaritmica).

In [4]:
anni = lambda k: int(k[:-1]) / (12 if k[-1] == "M" else 1)  # '3M' -> 0.25, '10Y' -> 10
curve = []
for area in ("US", "EUR", "IT", "DE", "FR", "ES"):
    cols = [c for c in df if c.startswith(area + " ") and c.split()[1][:-1].isdigit()]  # 'IT 10Y' sì, 'US IG spread' no
    s = df[cols].dropna()
    for quando in (0, 1, 3, 10):
        r = s.asof(s.index[-1] - pd.DateOffset(years=quando))
        curve += [{"Area": area, "Quando": f"{quando} anni fa" if quando else "oggi", "Anni": anni(c.split()[1]), "%": r[c]}
                  for c in cols]
px.line(pd.DataFrame(curve), x="Anni", y="%", color="Quando", facet_col="Area", facet_col_wrap=3, markers=True,
        log_x=True, height=650, title="Curva dei rendimenti").show()

## Inclinazione della curva (10 anni - 2 anni)
Sotto zero = curva invertita: storicamente anticipa un rallentamento.

In [5]:
fig = px.line(df.loc[DAL:, ["US 10Y-2Y", "EUR 10Y-2Y"]].dropna(how="all").ffill(), title="10Y - 2Y",
              labels={"value": "punti %", "Data": "", "variable": ""})
fig.add_hline(y=0, line_dash="dot")
fig.show()

## Spread verso il Bund e USA - area euro

In [6]:
linee(["BCE IT-DE 10Y", "BCE FR-DE 10Y", "BCE ES-DE 10Y"], "Spread verso il Bund dal 2000 (media mensile BCE)")
linee(["IT-DE 10Y", "FR-DE 10Y", "ES-DE 10Y", "US-EUR 10Y"], "Spread a 10 anni, giornaliero", dal="2023-11-01")

## Spread corporate (sopra i titoli di Stato)

In [7]:
linee(["US Baa spread"], "Spread aziende Baa sul Treasury 10 anni, dal 2000")
linee(["US IG spread", "US HY spread", "EUR HY spread"], "Spread corporate ICE (option-adjusted), ultimi 3 anni", dal=None)

## Riepilogo
Variazioni in punti base; percentile = posizione dell'ultimo valore negli ultimi 10 anni e in tutto lo storico (1 = mai così alto, 0 = mai così basso).

In [8]:
pd.read_excel(Path.cwd().parent / "data" / "riepilogo.xlsx")

,Serie,Dati al,Ultimo %,Var 1m (pb),Var 3m (pb),Var 1a (pb),Dal,Min,Max,Percentile 10a,Percentile storico
0,US 3M,2026-10-30,4.160,-4.000,34.000,24.000,2000,0.000,6.420,0.702,0.783
1,US 2Y,2026-10-30,4.560,-32.000,33.000,95.000,2000,0.090,6.930,0.893,0.866
2,US 5Y,2026-10-30,4.710,-38.000,33.000,99.000,2000,0.190,6.830,0.975,0.913
3,US 10Y,2026-10-30,4.940,-35.000,26.000,83.000,2000,0.520,6.790,0.992,0.913
4,US 30Y,2026-10-30,5.310,-33.000,10.000,66.000,2000,0.990,6.750,0.992,0.894
5,US IG spread,2026-11-02,0.940,9.000,15.000,14.000,2023,0.730,1.290,0.757,0.757
6,US HY spread,2026-11-02,3.430,33.000,58.000,49.000,2023,2.590,4.610,0.865,0.865
7,EUR HY spread,2026-11-02,3.560,27.000,91.000,75.000,2023,2.480,4.610,0.838,0.838
8,US Baa spread,2026-10-30,1.530,6.000,-11.000,-11.000,2000,1.360,6.160,0.083,0.031
9,EUR 3M,2026-11-02,2.573,5.150,19.264,67.132,2004,-0.930,4.325,0.810,0.798
